In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm
from collections import Counter

In [2]:
pth = os.path.abspath(os.path.join(os.getcwd(), ".."))
pth = os.path.abspath(os.path.join(pth, "data"))

In [15]:
# Lecture du fichier
df_train_train = pd.read_csv(pth + "/train-train.csv")
df_val = pd.read_csv(pth + "/val.csv")
df_test = pd.read_csv(pth + "/test.csv")

## 1) Import de Llama-3.1

In [4]:
import transformers
import torch

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.
Device set to use cpu


In [40]:
# Exemple d'utilisation
# Messages d'entrée
messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]


# Nombre de tokens à générer
max_tokens = 20

# Affichage de la barre de progression
with tqdm(total=max_tokens, desc="Generating", unit="token") as pbar:
    outputs = pipeline(messages, max_new_tokens=max_tokens)
    pbar.update(max_tokens)  # Mise à jour complète après génération

print(outputs[0]["generated_text"])

Generating: 100%|██████████| 20/20 [09:07<00:00, 27.38s/token]

[{'role': 'system', 'content': 'You are a pirate chatbot who always responds in pirate speak!'}, {'role': 'user', 'content': 'Who are you?'}, {'role': 'assistant', 'content': "Arrrr, me hearty! I be Captain Chat, a swashbucklin' pirate chat"}]


In [45]:
outputs[0]["generated_text"][-1]['content']

"Arrrr, me hearty! I be Captain Chat, a swashbucklin' pirate chat"

## 2) Data Augmentation

In [24]:
# Analyser les proportions des labels dans df_train_train
train_labels = df_train_train['label'].values
train_label_counts = Counter(train_labels)
train_total = len(train_labels)
train_proportions = {label: count / train_total for label, count in train_label_counts.items()}
print("Proportions dans l'entraînement :", train_proportions)

Proportions dans l'entraînement : {'out_of_scope': 0.2708333333333333, 'translate': 0.10416666666666667, 'book_flight': 0.08333333333333333, 'lost_luggage': 0.10416666666666667, 'travel_alert': 0.0625, 'travel_suggestion': 0.10416666666666667, 'carry_on': 0.10416666666666667, 'book_hotel': 0.08333333333333333, 'flight_status': 0.08333333333333333}


In [25]:
val_labels = df_val['label'].values
val_label_counts = Counter(val_labels)
val_total = len(val_labels)
val_proportions = {label: count / val_total for label, count in val_label_counts.items()}
print("Proportions dans la validation :", val_proportions)

Proportions dans la validation : {'out_of_scope': 0.3333333333333333, 'flight_status': 0.08333333333333333, 'book_hotel': 0.08333333333333333, 'lost_luggage': 0.08333333333333333, 'translate': 0.08333333333333333, 'travel_alert': 0.08333333333333333, 'travel_suggestion': 0.08333333333333333, 'book_flight': 0.08333333333333333, 'carry_on': 0.08333333333333333}


In [26]:
test_labels = df_test['label'].values
test_label_counts = Counter(test_labels)
test_total = len(test_labels)
test_proportions = {label: count / test_total for label, count in test_label_counts.items()}
print("Proportions dans le test :", test_proportions)

Proportions dans le test : {'out_of_scope': 0.26666666666666666, 'book_hotel': 0.13333333333333333, 'carry_on': 0.13333333333333333, 'lost_luggage': 0.06666666666666667, 'travel_alert': 0.06666666666666667, 'book_flight': 0.06666666666666667, 'translate': 0.06666666666666667, 'flight_status': 0.06666666666666667, 'travel_suggestion': 0.13333333333333333}


In [ ]:
# Fonction pour augmenter un exemple de texte
def augment_text(text, num_augments):
    augmented_texts = []
    contexte = '''
                Tu es un assistant capable de générer de nouveaux textes à partir d'un texte original en modifiant 
                certains détails tout en conservant l'idée principale.
                L'objectif est d'enrichir le dataset pour l'entraînement d'un modèle en introduisant plus de diversité.
                Pour cela, tu peux varier les détails spécifiques (ex. : noms, lieux, chiffres, objets), 
                changer la structure et la formulation des phrases, ou encore traduire en anglais, allemand, italien,
                portugais ou espagnol.
                Tu es libre d'ajouter ou modifier des informations, reformuler les phrases, ou les rendre plus concises ou 
                détaillées selon le besoin.

                Exemples de génération de texte :

                    - Si je vais à Evans, que dois-je faire ? → Qu'est-ce que je peux faire à Lille ?
                    - Quand mon vol atterrira-t-il ? → Is my flight at 6:00?
                    - Y a-t-il une limite de poids de bagage cabine ? → Combien de kilos puis-je mettre dans mon bagage cabine ?
                    - Quelle est la température à Paris ? → What is the weather like in London?
                    - Quel est le prix d'un billet pour le musée ? → How much does a ticket to the museum cost?


                Dans ces exemples, certains détails sont modifiés (lieux, heures, objets),
                et certaines phrases sont traduites ou reformulées pour enrichir la diversité des formulations.

                Tu ne dois générer que la phrase augmentée, sans explication ni mise en contexte ni de phrase comme : "Voici quelques variations"

            
                '''
    # Affichage de la barre de progression pour chaque augmentation
    with tqdm(total=num_augments, desc="Generating", unit="example") as pbar:
        for _ in range(num_augments):
            messages = [
                {"role": "system", "content": contexte},
                {"role": "user", "content": text},]
            outputs = pipeline(messages)
            augmented_texts.append(outputs[0]["generated_text"][-1]['content'])
            pbar.update(1)  # Mise à jour de la barre de progression

    return augmented_texts 

In [ ]:
# Fonction pour augmenter les textes d'un DataFrame, respectant les proportions des labels
def augment_dataframe(df, num_augments, max_tokens=20):
    augmented_df = pd.DataFrame(columns=df.columns)  # DataFrame pour stocker les augmentations
    
    # Affichage de la barre de progression pour les lignes du DataFrame
    with tqdm(total=df.shape[0], desc="Augmenting Data", unit="row") as pbar:
        for label in df['label'].unique():  # Parcours des labels uniques
            label_data = df[df['label'] == label]  # Sous-DataFrame pour chaque label
            
            for _, row in label_data.iterrows():
                # Texte à augmenter
                original_text = row['text'] 
                
                # Générer des augmentations
                augmented_texts = augment_text(original_text, num_augments=num_augments, max_tokens=max_tokens)
                
                # Crée une liste pour stocker les nouvelles lignes sous forme de dictionnaire
                new_rows = []
                
                # Ajouter les nouveaux textes dans new_rows avec une valeur fixe pour 'label'
                for augmented_text in augmented_texts:
                    new_rows.append({'text': augmented_text, 'label': label})
                
                # Ajouter les nouvelles lignes dans augmented_df
                augmented_df = pd.concat([augmented_df, pd.DataFrame(new_rows)], ignore_index=True)
            
            pbar.update(1)  # Mise à jour de la barre de progression
    
    # Concaténer augmented_df avec df
    final_df = pd.concat([df, augmented_df], ignore_index=True)
    
    return final_df


In [38]:
augmented_train_train = augment_dataframe(df_train_train, num_augments=20, max_tokens=20)
augmented_val = augment_dataframe(df_val, num_augments=20, max_tokens=20)
augmented_test = augment_dataframe(df_test, num_augments=20, max_tokens=20)

Augmenting Data:   0%|          | 0/48 [00:00<?, ?row/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Augmenting Data:   0%|          | 0/48 [11:03<?, ?row/s]


KeyboardInterrupt: 